In [ ]:
import sys
import pickle
import shap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append('../')
from utils.shap_utils import shap_dotplot_by_risk


# Get the data

In [ ]:
## Fill in your own settings here!!

# Insert your own WSI annotations here
wsi_annotations = ['Tumor', '--', 'Tissue edge: stroma > tumor', 'Tumor', 'Adipose > stroma', '--', 'Stroma > adipose', 'Adipose > stroma', 'Tumor: solid pattern, abundant cytoplasm', 'Tumor >> adipose', 'Stroma > tumor > immune cells', 'Tumor > stroma > immune cells', 'Normal ducts/lobules', 'Stroma > adipose', 'Edge stainings, blood vessels', 'Stroma >  immune cells']

# Data type
type = 'brca'

# Fold
fold = 2

# Name of experiment/model
exp_name = "DIMAFx"


In [ ]:
# Loading pathways names and initializing cluster identities
hallmarks = pd.read_csv('../data/data_files/hallmarks_signatures.csv')
hallmarks = np.array([' '.join(x[9:].split('_')) for x in hallmarks.columns])
hallmarks_ids = np.array([f'R{i}' for i, item in enumerate (hallmarks)])
hallmarks_names = np.array([f'R{i}: {item}' for i, item in enumerate (hallmarks)])

cluster_ids = np.array([f'W{i}' for i, item in enumerate (wsi_annotations)])
cluster_names = np.array([f'W{i}: {item}' for i, item in enumerate (wsi_annotations)])

all_feats_names = [f"{name}" for name in cluster_names] + [f"{name}" for name in hallmarks_names]
all_short_feat_names = np.concatenate((hallmarks_ids, cluster_ids))

In [ ]:
# Obtain shap values
shap_results_fold_dir = f'../results/dss_survival_{type}/{exp_name}/Fold_{fold}/post_training/shap/modal/shap_all_test.pkl'
shap_dict = pickle.load(open(shap_results_fold_dir, 'rb'))     
shap_values = shap_dict['shap values']
all_feats_shap = np.sum(shap_values, axis=2)

# Obtain risks
data_results_path = f"../results/dss_survival_{type}/{exp_name}/Fold_{fold}/post_training/predicted_risk_scores_test.pkl"
scores_results = pickle.load(open(data_results_path, 'rb'))   

# Plot shap plot vs risk score

In [ ]:
shap_dotplot_by_risk(all_feats_shap, all_feats_names, scores_results['Risk scores'])

# Plot SHAP of single sample

In [ ]:
def get_plot_single_sample(sample, shap_dict, all_feats_names, max_display=15):
    """ Plot SHAP values for a single sample."""
    # Get shap values of this sample
    shap_values = shap_dict['shap values']
    sample_index = shap_dict['Samples'].index(sample)
    shap_of_one_sample = np.expand_dims(np.sum(shap_values[sample_index], axis=-1),axis=0)

    # Plot shap
    sample_expl = shap.Explanation(values=shap_of_one_sample, feature_names=all_feats_names)
    shap.plots.bar(sample_expl[0], max_display=max_display)
    plt.show()

In [ ]:
# We can also plot the local multimodal feature importance plot of one sample
# Take a test id of the specified fold
id = 'TCGA-3C-AALK'
get_plot_single_sample(id, shap_dict, all_feats_names)